In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDateTimeDF.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    text.replace("  "," ")
    return text

for date, row in df_articles.T.iteritems():
    print("The date is:"+str(date))
    # try:
    sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = clean_text(sentence)

    ss = sid.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
            # df.at[date, 'sentiment_complementary'] = 100 - 100*(ss['pos'] / (ss['pos'] + ss['neg']))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
        
    # except TypeError:
    #     #print("sentence is")
    #     #print(sentence)
    #     #print(df_vix.loc[date, 'articles'])
    #     print("ERROR:"+sentence)
    #     df.at[date, 'compound'] = 0
    #     df.at[date, 'neg'] = 0
    #     df.at[date, 'neu'] = 0
    #     df.at[date, 'pos'] = 0
    #     # df.at[date, 'sentiment'] = 0
    #     # df.at[date, 'sentiment_complementary'] = 0

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))



In [ ]:
df = df.join(df_articles)

In [ ]:
df["Date"] = df.index

In [ ]:
df

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesTime.pkl")